In [1]:
#Install Libraries
pip install langchain tqdm sentence-transformers faiss-cpu transformers sentencepiece transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Install Libraries
pip install  huggingface_hub[hf_xet]   hf_xet

In [3]:
#Install Libraries
pip install evaluate scikit-learn bert-score rouge_score absl-py


Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Import Libraries

import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm  
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration
import faiss
from evaluate import load
import numpy as np
import time


In [4]:
#Load Dataset
path='career_counselling_documents_500.csv'
docs= pd.read_csv(path)
docs

,id,category,document
0,a5db8ec9-a4f3-4bf2-bce5-950462bbdc0e,Courses & Programs,The Master of Science in AI program offers stu...
1,b91302b4-c513-496e-9b10-428e08592cff,Industry Trends,The healthcare industry is undergoing signific...
2,8ad2eb4e-be75-4651-8c54-7b626e853923,University Admissions,Applying to international universities involve...
3,1511d146-db6f-4afe-b2ea-9c320a911a89,Skills & Certifications,"In today's job market, skills like graphic des..."
4,12673e4d-2e47-486b-8e74-881dff738c00,Entrance Exams,Entrance exams like the SAT assess a student's...
...,...,...,...
495,f560ffc9-58ff-4b7b-880f-51ff6fa4477b,Skills & Certifications,"In today's job market, skills like economics a..."
496,5616f345-d77e-4770-8af3-501c5b885607,Entrance Exams,Entrance exams like the GATE assess a student'...
497,caa526b5-9fea-4bc7-be7e-62d72b4a56a1,Entrance Exams,Entrance exams like the GRE assess a student's...
498,69ee0f2a-2449-45d0-aaf8-afbf09624a27,Courses & Programs,The Diploma in Graphic Design program offers s...


In [ ]:
#Chunking
# Initialize the text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", "!", "?"]
)

# Initialize list for chunks
chunks = []

# Iterate through each document with a progress bar
for _, row in tqdm(docs.iterrows(), total=len(docs), desc="Chunking documents"):
    doc_id = row.get("id")
    category = row.get("category")
    text = str(row.get("document", "")).strip()
    
    # Skip empty documents
    if not text:
        continue

    # Split the text into chunks
    parts = splitter.split_text(text)
    
    for i, part in enumerate(parts):
        chunks.append({
            "id": f"{doc_id}_{i}",
            "category": category,
            "chunk": part
        })

# Convert list of chunks to a DataFrame
chunk_df = pd.DataFrame(chunks)

# Save to CSV
chunk_df.to_csv("career_chunks.csv", index=False)

print(f" Chunking complete: {len(chunk_df)} chunks saved to 'career_chunks.csv'")


Chunking documents: 100%|██████████| 500/500 [00:00<00:00, 3477.61it/s]

✅ Chunking complete: 841 chunks saved to 'career_chunks.csv'


In [6]:
#Vector Embedding
# Load the chunked data
df = pd.read_csv("career_chunks.csv")

# Load a sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2") 

# Create dense vector embeddings for each chunk
document_texts = df['chunk'].tolist()
document_embeddings = model.encode(document_texts, show_progress_bar=True)
print("Chunks Convert into vertor form :-\n",document_embeddings)
print("Vector Rows & Columns:-\n",document_embeddings.shape)


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Chunks Convert into vertor form :-
 [[-0.01638853 -0.00870151  0.05265848 ...  0.01891424 -0.06158472
  -0.00446476]
 [ 0.04258545  0.00048218  0.03052215 ...  0.05512461  0.03094077
   0.00217226]
 [-0.03002284 -0.04613508 -0.05133712 ... -0.05158173  0.07062405
  -0.00357103]
 ...
 [ 0.04258545  0.00048218  0.03052215 ...  0.05512461  0.03094077
   0.00217226]
 [-0.02351535 -0.04173231  0.0214807  ... -0.07476959  0.02404717
   0.0392506 ]
 [ 0.0149943   0.01533544 -0.06767237 ...  0.027764    0.03201963
   0.05459788]]
Vector Rows & Columns:-
 (841, 384)


In [7]:
#Cosine Similarity
# Define vector dimension
dimension = document_embeddings.shape[1]

# Create a FAISS index (L2 or cosine-based)
index = faiss.IndexFlatIP(dimension)  # Cosine similarity (inner product)

# Normalize vectors for cosine similarity
faiss.normalize_L2(document_embeddings)

# Add embeddings to FAISS index
index.add(np.array(document_embeddings))

from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_k(query, k=5, similarity_threshold=0.95):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    # Retrieve top 20 results (so we can filter later)
    distances, indices = index.search(query_embedding, 20)
    raw_results = df.iloc[indices[0]].copy()
    raw_results["score"] = distances[0]
    
    # Filter out semantically similar chunks
    selected = []
    used_embeddings = []

    for _, row in raw_results.iterrows():
        chunk_embedding = model.encode([row["chunk"]])
        faiss.normalize_L2(chunk_embedding)

        if all(cosine_similarity(chunk_embedding, emb)[0][0] < similarity_threshold for emb in used_embeddings):
            selected.append(row)
            used_embeddings.append(chunk_embedding)

        if len(selected) == k:
            break

    return pd.DataFrame(selected)

print("Working")

Working


In [8]:
#Example 1:-
query = "How do I prepare for a data science career?"
top_docs = retrieve_top_k(query, k=5)

for i, row in top_docs.iterrows():
    print(f"\nResult {i+1} (Score: {row['score']:.4f}):")
    print(row["chunk"])



Result 768 (Score: 0.7251):
A career in data science can be both rewarding and challenging. With the right skills and experience, professionals in this area can find opportunities across industries

Result 162 (Score: 0.5484):
The Diploma in Graphic Design program offers students an in-depth understanding of data science. It typically lasts 6 months and covers subjects such as biochemistry, machine learning, and economics. Graduates are equipped to enter fields like data analyst and data analyst

Result 418 (Score: 0.5381):
The Master of Science in AI program offers students an in-depth understanding of data science. It typically lasts 3 years and covers subjects such as graphic design, graphic design, and biochemistry. Graduates are equipped to enter fields like research assistant and data analyst

Result 560 (Score: 0.5090):
. Many students start with a degree in BSc and gradually move into specialized roles like data analyst, research assistant, or even data analyst. Soft skills su

In [10]:
#Generator Module
# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-base", legacy=False)
generator_model = T5ForConditionalGeneration.from_pretrained("t5-base")

# Generator function
def generate_answer_t5(user_query, context_chunks, max_input_len=512, max_output_len=150):
    context = " ".join(context_chunks)  # Combine chunks into a single context
    prompt = f"question: {user_query} context: {context}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len,
        padding=True
    )

    output_ids = generator_model.generate(
        inputs["input_ids"],
        max_length=max_output_len,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)




In [ ]:
# Example 1:-
user_query = "What are the best career options for computer science graduates?"
context_chunks = df['chunk'].tolist() 

# Generate answer
response = generate_answer_t5(user_query, context_chunks)

# Output
print(" Query:", user_query)
print("\n Generated Answer:\n", response)

🧠 Query: What are the best career options for computer science graduates?

✅ Generated Answer:
 UX designer and cybersecurity expert


In [ ]:
# Example 2:-
user_query2 = "What skills are learn for data science students?"
context_chunks2 = df['chunk'].tolist()  

# Generate answer
response2 = generate_answer_t5(user_query2, context_chunks2)

# Output
print(" Query:", user_query2)
print("\n Generated Answer:\n", response2)

🧠 Query: What skills are learn for data science students?

✅ Generated Answer:
 in-depth understanding of data science


In [ ]:
#System Integration

# Load Data and Models
df = pd.read_csv("career_chunks.csv")
chunks = df['chunk'].tolist()

# Sentence embedding model for dense retrieval
retriever_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generator model and tokenizer (T5)
tokenizer = T5Tokenizer.from_pretrained("t5-base", legacy=False)
generator_model = T5ForConditionalGeneration.from_pretrained("t5-base")

# Build FAISS Index
document_embeddings = retriever_model.encode(chunks, normalize_embeddings=True)
index = faiss.IndexFlatIP(document_embeddings.shape[1])
index.add(document_embeddings)

# Dense Retriever Function
def retrieve_top_k(query, k=5):
    query_embedding = retriever_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_embedding), k)
    top_chunks = [chunks[i] for i in indices[0]]
    top_scores = scores[0]
    return list(zip(top_chunks, top_scores))

# Generator Function
def generate_answer_t5(user_query, context_pairs, max_input_len=512, max_output_len=150):
    context = " ".join([chunk for chunk, _ in context_pairs])
    prompt = f"question: {user_query} context: {context}"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_len, padding=True)

    output_ids = generator_model.generate(
        inputs["input_ids"],
        max_length=max_output_len,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Inference Pipeline
def answer_pipeline(query, top_k=1):
    top_chunks = retrieve_top_k(query, k=top_k)
    answer = generate_answer_t5(query, top_chunks)

    # Display results with citation
    print(" Query:", query)
    print("\n Answer:\n", answer)
    print("\n Context used:")
    for i, (chunk, score) in enumerate(top_chunks):
        print(f"\n{i+1}. [Score: {score:.4f}]\n{chunk}")

# Example
query = "What are the best career options for data science graduates?"
answer_pipeline(query)


🧠 Query: What are the best career options for data science graduates?

✅ Answer:
 rewarding and challenging

📚 Context used:

1. [Score: 0.7992]
A career in data science can be both rewarding and challenging. With the right skills and experience, professionals in this area can find opportunities across industries


In [ ]:
#Evaluation

# Load metrics
rouge = load("rouge")
bertscore = load("bertscore")

#  Sample data
eval_data = [
    {
        "query": "What are good careers for CS graduates?",
        "retrieved_docs": [
            "Computer science students become software engineers.",
            "AI and cybersecurity are top fields."
        ],
        "ground_truth": "Computer science graduates can become software engineers, AI specialists, or cybersecurity experts.",
        "generated_answer": "Computer science graduates can pursue careers in AI, software development, and cybersecurity."
    }
]

#  Retrieval metrics
def compute_retrieval_metrics(eval_data, k=5):
    recalls = []
    reciprocal_ranks = []

    for sample in eval_data:
        relevant = sample['ground_truth'].lower()
        retrieved = [doc.lower() for doc in sample['retrieved_docs']]

        match_found = False
        for idx, doc in enumerate(retrieved[:k]):
            if relevant[:50] in doc or doc in relevant:
                recalls.append(1)
                reciprocal_ranks.append(1 / (idx + 1))
                match_found = True
                break

        if not match_found:
            recalls.append(0)
            reciprocal_ranks.append(0)

    return np.mean(recalls), np.mean(reciprocal_ranks)

#  Generation metrics
def compute_generation_metrics(eval_data):
    predictions = [ex['generated_answer'] for ex in eval_data]
    references = [ex['ground_truth'] for ex in eval_data]

    rouge_results = rouge.compute(predictions=predictions, references=references)
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")

    return {
        "ROUGE": rouge_results,
        "BERTScore-F1": float(np.mean(bert_results['f1']))
    }

#  Dummy generator
def dummy_generator(query, context):
    return context[0]

#  Latency
def measure_latency(generate_func, samples):
    total_time = 0
    for sample in samples:
        start = time.time()
        _ = generate_func(sample["query"], sample["retrieved_docs"])
        total_time += time.time() - start
    return total_time / len(samples)

#  Run Evaluation
recall_at_k, mrr = compute_retrieval_metrics(eval_data)
print(f"\n📥 Retrieval Metrics:\nRecall@k: {recall_at_k:.2f}\nMRR: {mrr:.2f}")

gen_metrics = compute_generation_metrics(eval_data)
print("\n📝 Generation Metrics:")
print("ROUGE:", gen_metrics["ROUGE"])
print("BERTScore F1:", f"{gen_metrics['BERTScore-F1']:.4f}")

latency = measure_latency(dummy_generator, eval_data)
print(f"\n⚡ Avg Latency per Generation: {latency:.4f} seconds")



📥 Retrieval Metrics:
Recall@k: 0.00
MRR: 0.00


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

OSError: Can't load the model for 'roberta-large'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'roberta-large' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.